Jai Shree Ram

In [ ]:
%pip install qiskit
%pip install qiskit_aer
%pip install "qiskit[visualization]"

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(1)
#qc.x, t,.....
qc.h(0)
qc.draw("mpl")

In [ ]:
qc = QuantumCircuit(3)
#qc.swap(0,1)
#qc.cx(0, 1)
qc.ccx(0,1,2)
qc.draw("mpl")
#qc.decompose().draw("mpl")

In [ ]:
qc = QuantumCircuit(
    1, 1
)  # the second number is the number of classical bits in the circuit
qc.measure(0, 0)
qc.draw("mpl")

In [ ]:
# qubits: a, b, sum, carry
qc = QuantumCircuit(4)

# Choose values for A and B:
a = 0
b = 0

# Prepare A and B qubits according to selected values:
if a:
    qc.x(0)
if b:
    qc.x(1)

# XOR (sum) into qubit 2
qc.cx(0, 2)
qc.cx(1, 2)

# AND (carry) into qubit 3
qc.ccx(0, 1, 3)  # a AND b

# measure
qc.measure_all()


qc.draw("mpl")



In [ ]:
# Load the backend sampler
from qiskit.primitives import BackendSamplerV2

# Load the Aer simulator and generate a noise model based on the currently-selected backend.
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel

noise_model = NoiseModel.from_backend(backend)

# Define a simulator using Aer, and use it in Sampler.
backend_sim = AerSimulator(noise_model=noise_model)
sampler_sim = BackendSamplerV2(backend=backend_sim)

# Alternatively, load a fake backend with generic properties and define a simulator.
# backend_gen = GenericBackendV2(num_qubits=18)
# sampler_gen = BackendSamplerV2(backend=backend_gen)

job = sampler.run([qc_isa], shots=100)
# job = sampler_sim.run([qc_isa]) # uncomment if you want to run on a simulator
res = job.result()
counts = res[0].data.meas.get_counts()

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.primitives import BackendSamplerV2
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel

# 1. Initialize the service to access IBM Quantum backends
# (Make sure you have saved your IBM Quantum API token to your machine previously)
service = QiskitRuntimeService()

# 2. Select a real backend to mimic (e.g., 'ibm_brisbane' or any available system)
backend = service.least_busy(simulator=False, operational=True)
print(f"Mimicking noise profile from real device: {backend.name}")

# 3. Extract the noise model from that specific real backend
noise_model = NoiseModel.from_backend(backend)

# 4. Define your high-performance local Aer Simulator using that noise model
backend_sim = AerSimulator(noise_model=noise_model)

# 5. Wrap the simulator in the updated V2 Sampler primitive
sampler_sim = BackendSamplerV2(backend=backend_sim)

# 6. Run your transpiled circuit (qc_isa) on the noisy local simulator
# Note: Changed 'sampler' to 'sampler_sim' to match your definition!
job = sampler_sim.run([qc_isa], shots=100)
res = job.result()

# 7. Extract the measurement counts out of the V2 Primitive PubResult structure
# (In V2, result data is grouped by the 'classical register' name, usually 'meas')
counts = res[0].data.meas.get_counts()

print("Execution successful! Sample counts:", counts)

If your goal is a 100% offline, local simulation environment without connecting to cloud hardware at all, you want to ditch QiskitRuntimeService entirely.
Instead, you can use Qiskit's built-in GenericBackendV2. It mocks up a real physical quantum computer's chip layout, gate errors, and read-out noise patterns entirely inside your local machine's RAM.

In [ ]:
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.primitives import BackendSamplerV2
from qiskit.visualization import plot_histogram

# 1. Initialize an offline mock 7-qubit hardware backend
backend_local_hardware = GenericBackendV2(num_qubits=7)
print(f"Target Backend: {backend_local_hardware.name}")

# 2. TRANSPILE: Compile 'qc' into 'qc_isa' so the hardware layout understands it
pm = generate_preset_pass_manager(optimization_level=1, backend=backend_local_hardware)
qc_isa = pm.run(qc)
print("Circuit transpilation to Instruction Set Architecture (ISA) complete.")

# 3. Initialize the V2 Sampler Primitive targeting our local engine
sampler_sim = BackendSamplerV2(backend=backend_local_hardware)

# 4. Run the transpiled circuit locally with 100 shots
job = sampler_sim.run([qc_isa], shots=100)
res = job.result()

# 5. Extract the measurement result dictionary out of the V2 PubResult format
# res[0] grabs the first pub; .data.meas pulls from the 'meas' register we created
counts = res[0].data.meas.get_counts()

print("\n--- Simulation Results ---")
print("Format: 'Carry Sum B A' (Read right-to-left due to Qiskit ordering)")
print("counts =",counts)
plot_histogram(counts)
